# Phase 1D.3 Colab run

Use a fresh Tesla T4 runtime. This notebook stops on environment or wheel provenance mismatches. It does not create a public tunnel. Save cell outputs as evidence.

In [ ]:
import hashlib, json, os, subprocess
from pathlib import Path
REPO_URL='https://github.com/MoeMo-s-Team/DACN_HeThongHoTroTuhoc_PhanHeNguoiDung.git'
REVIEWED_COMMIT=os.environ.get('PHASE1D3_REVIEWED_COMMIT')
if not REVIEWED_COMMIT or len(REVIEWED_COMMIT)!=40: raise RuntimeError('Set PHASE1D3_REVIEWED_COMMIT to the reviewed 40-character commit.')
REPO_BASE=Path('/content/phase-1d3-repo-'+REVIEWED_COMMIT[:8])
REPO=REPO_BASE
suffix=0
while REPO.exists():
    suffix+=1
    REPO=Path(f'{REPO_BASE}-{suffix}')
os.environ['PHASE1D3_REPO']=str(REPO)
clone=subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO)],capture_output=True,text=True)
if clone.returncode: raise RuntimeError(f'git clone failed ({clone.returncode}):\n{clone.stderr}')
subprocess.run(['git','checkout','--detach',REVIEWED_COMMIT],cwd=REPO,check=True)
actual=subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
dirty=subprocess.run(['git','status','--porcelain','--untracked-files=all'],cwd=REPO,capture_output=True,text=True,check=True).stdout
if actual!=REVIEWED_COMMIT or dirty: raise RuntimeError('Reviewed source checkout mismatch or dirty tree.')
proof={'verified':True,'commit':actual,'repo':str(REPO),'tree':subprocess.run(['git','rev-parse','HEAD^{tree}'],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()}
proof_path=Path('/content/dacn-source-proof.json'); proof_path.write_text(json.dumps(proof,indent=2)+'\n')
print(proof)


In [ ]:
import sys
sys.path.insert(0,str(REPO))
os.chdir(REPO)
from ai.colab.torch_provenance import verify_source_proof
proof_commit=verify_source_proof('/content/dacn-source-proof.json',REPO)
if proof_commit!=REVIEWED_COMMIT: raise RuntimeError('BLOCKED: source proof differs from requested reviewed commit.')
print('Verified clean source commit:',proof_commit)


In [ ]:
# Snapshot and hash every system site-packages file before dependency work.
import json, platform, sys, subprocess, hashlib, site
sys.dont_write_bytecode=True
os.environ['PYTHONDONTWRITEBYTECODE']='1'
import torch
from pathlib import Path
from ai.colab.preflight import system_site_packages_inventory
from ai.colab.torch_provenance import _torch_file_inventory
roots=[root for root in site.getsitepackages() if Path(root).is_dir()]
if site.ENABLE_USER_SITE and Path(site.getusersitepackages()).is_dir(): roots.append(site.getusersitepackages())
if not roots: raise RuntimeError('BLOCKED: no system site-packages directories could be inventoried.')
torch_root=Path(torch.__file__).parent.parent
snapshot={'python':platform.python_version(),'pythonExecutable':sys.executable,'systemPrefix':sys.prefix,'systemBasePrefix':sys.base_prefix,
 'torchRoot':str(torch_root),'torch':torch.__version__,'cuda':torch.version.cuda,
 'pipFreeze':subprocess.run([sys.executable,'-B','-m','pip','freeze'],capture_output=True,text=True,check=True).stdout.splitlines(),
 'torchFiles':_torch_file_inventory(torch_root,__import__('importlib.metadata').metadata.distribution('torch')),'systemSitePackagesRoots':roots,'systemSitePackagesFiles':system_site_packages_inventory(roots),
 'gpu':subprocess.run(['nvidia-smi','--query-gpu=name,uuid,memory.total','--format=csv,noheader'],capture_output=True,text=True,check=True).stdout.strip()}
summary={key:value for key,value in snapshot.items() if key not in ('torchFiles','systemSitePackagesFiles','pipFreeze')}
summary.update({'torchFilesHashed':len(snapshot['torchFiles']),'systemSitePackagesFilesHashed':len(snapshot['systemSitePackagesFiles']),'systemPipFreezeCount':len(snapshot['pipFreeze'])})
print(json.dumps(summary,indent=2))
if not snapshot['python'].startswith('3.13.') or snapshot['torch']!='2.11.0+cu130' or snapshot['cuda']!='13.0' or 'Tesla T4' not in snapshot['gpu']:
    raise RuntimeError('BLOCKED: Python/Torch/CUDA/T4 candidate mismatch; leave system packages unchanged.')
Path('/content/dacn-colab-before.json').write_text(json.dumps(snapshot,indent=2)+'\n')

In [ ]:

from pathlib import Path
import json

snapshot_path = Path("/content/dacn-colab-before.json")
repo = REPO

assert snapshot_path.is_file(), "Snapshot chưa được lưu"

data = json.loads(snapshot_path.read_text())

print("SNAPSHOT: PASS")
print("Python:", data["python"])
print("Torch:", data["torch"])
print("CUDA:", data["cuda"])
print("GPU:", data["gpu"].split(",")[0])
print("Torch files hashed:", len(data["torchFiles"]))
print("System site-packages files hashed:", len(data["systemSitePackagesFiles"]))

print("\nDEPENDENCY GATE:")
print("External hash lock:", Path("/content/ai-requirements.lock").exists())
print("Official Torch wheel evidence:", Path("/content/dacn-torch-wheel.json").exists())
print("Installed venv provenance:", Path("/content/torch-wheel-provenance.json").exists())


In [ ]:
from ai.colab.torch_provenance import download_official_wheel, verify_source_proof
if verify_source_proof('/content/dacn-source-proof.json',REPO)!=REVIEWED_COMMIT: raise RuntimeError('BLOCKED: source proof changed before wheel download.')
wheel_evidence=download_official_wheel('/content/dacn-torch-wheels')
wheel_evidence.update({'sourceCommit':REVIEWED_COMMIT,'sourceProofSha256':hashlib.sha256(Path('/content/dacn-source-proof.json').read_bytes()).hexdigest()})
Path('/content/dacn-torch-wheel.json').write_text(json.dumps(wheel_evidence,indent=2)+'\n')
print(json.dumps(wheel_evidence,indent=2))

## Dependency lock gate

After the official wheel download cell passes, paste this block into a new Colab notebook code cell. Its `%%bash` magic inherits the reviewed checkout path from the notebook kernel. Create a **separate resolver** and full hash lock outside the source checkout. Torch is a direct requirement; include its official SHA-256 in the lock:

```bash
%%bash
set -euo pipefail
for name in ${!PIP_@}; do unset "$name"; done
unset PYTHONPATH PYTHONHOME PYTHONUSERBASE PYTHONEXECUTABLE PYTHONSTARTUP PYTHONNOUSERSITE VIRTUAL_ENV
export PIP_CONFIG_FILE=/dev/null
COLAB_PYTHON="$(python3 -c 'import sys; print(sys.executable)')"
"$COLAB_PYTHON" -c 'import sys; assert sys.version_info[:2] == (3, 13)'
PIP_HELP="$($COLAB_PYTHON -m pip --help)"
if [[ "$PIP_HELP" != *"--python <python>"* ]]; then echo "BLOCKED: Colab system pip must support --python to seed the resolver without ensurepip." >&2; exit 1; fi
RESOLVER="$(mktemp -d /content/phase1d3-lock-resolver.XXXXXX)"
"$COLAB_PYTHON" -m venv --without-pip "$RESOLVER"
"$COLAB_PYTHON" -m pip --python "$RESOLVER" install --index-url https://pypi.org/simple pip
"$RESOLVER/bin/python" -m pip --isolated install --index-url https://pypi.org/simple pip-tools==7.5.2
"$RESOLVER/bin/pip-compile" --index-url https://pypi.org/simple --extra-index-url https://download.pytorch.org/whl/cu130 --generate-hashes --allow-unsafe --output-file /content/ai-requirements.lock --constraint "$PHASE1D3_REPO/ai/constraints-colab.txt" "$PHASE1D3_REPO/ai/requirements.in"
```

Keep resolver output and lock under `/content`, separate from the clean reviewed source checkout. The following dry-run is the last pre-install gate; it must use a new venv without system-site packages and accept only the verified Torch wheel hash. The install cell will stop unless every gate passes.


In [ ]:
from ai.colab.preflight import system_site_packages_inventory, validate_venv_install_plan, lock_contains_torch_hash
from ai.colab.torch_provenance import download_official_wheel, verify_source_proof
VENV_ROOT=Path('/content/dacn-venv'); VENV_PY=str(VENV_ROOT/'bin/python')
def isolated_venv_env():
    env={key:value for key,value in os.environ.items() if not key.startswith('PIP_') and key not in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE','VIRTUAL_ENV')}
    env.update({'PIP_CONFIG_FILE':os.devnull,'PYTHONNOUSERSITE':'1','PYTHONDONTWRITEBYTECODE':'1'})
    return env
PIP_ENV=isolated_venv_env()
source_proof_path=Path('/content/dacn-source-proof.json'); wheel_path=Path('/content/dacn-torch-wheel.json'); lock=Path('/content/ai-requirements.lock'); plan=Path('/content/dacn-install-plan.json')
if VENV_ROOT.exists(): raise RuntimeError('BLOCKED: target venv already exists; use a clean runtime to prove it is fresh.')
if not source_proof_path.is_file() or not wheel_path.is_file() or not lock.is_file(): raise RuntimeError('BLOCKED: source proof, verified wheel evidence and external resolver lock are required.')
system_pip_help=subprocess.run([sys.executable,'-m','pip','--help'],capture_output=True,text=True,check=True,env=PIP_ENV).stdout
if '--python <python>' not in system_pip_help: raise RuntimeError('BLOCKED: Colab system pip must support --python to seed the isolated venv without ensurepip.')
subprocess.run([sys.executable,'-m','venv','--without-pip',str(VENV_ROOT)],check=True,env=PIP_ENV)
subprocess.run([sys.executable,'-m','pip','--python',str(VENV_ROOT),'install','--index-url','https://pypi.org/simple','pip'],check=True,env=PIP_ENV)
venv_probe="import importlib.metadata as m,json,sys,pip; from pathlib import Path; c=(Path(sys.prefix)/'pyvenv.cfg').read_text(); packages={d.metadata['Name'].lower().replace('_','-'):d.version for d in m.distributions() if d.metadata.get('Name')};\ntry: t=m.version('torch')\nexcept m.PackageNotFoundError: t=None\nprint(json.dumps({'pythonExecutable':sys.executable,'python':__import__('platform').python_version(),'prefix':sys.prefix,'basePrefix':sys.base_prefix,'torch':t,'installedPackages':packages,'pipRoot':str(Path(pip.__file__).resolve().parent),'pyvenvCfg':c}))"
venv_info=json.loads(subprocess.run([VENV_PY,'-B','-c',venv_probe],capture_output=True,text=True,check=True,env=PIP_ENV).stdout)
if venv_info['prefix']==venv_info['basePrefix'] or venv_info['python']!=snapshot['python'] or not Path(venv_info['pipRoot']).is_relative_to(Path(venv_info['prefix'])) or 'pip' not in venv_info['installedPackages'] or 'include-system-site-packages = false' not in venv_info['pyvenvCfg'].lower() or venv_info['torch'] is not None: raise RuntimeError('BLOCKED: target must be a fresh isolated venv with venv-local pip and no Torch installed.')
source_proof=json.loads(source_proof_path.read_text()); wheel=json.loads(wheel_path.read_text()); actual_commit=verify_source_proof(source_proof_path,REPO); source_hash=hashlib.sha256(source_proof_path.read_bytes()).hexdigest()
if not source_proof.get('verified') or actual_commit!=REVIEWED_COMMIT or source_proof.get('commit')!=actual_commit or wheel.get('sourceCommit')!=actual_commit or wheel.get('sourceProofSha256')!=source_hash: raise RuntimeError('BLOCKED: reviewed source proof and wheel evidence do not match.')
official_wheel=download_official_wheel(Path(wheel.get('wheelPath','')).parent)
if official_wheel['wheelPath']!=wheel.get('wheelPath') or official_wheel['sha256']!=wheel.get('sha256') or not lock_contains_torch_hash(lock.read_text(),official_wheel['sha256']): raise RuntimeError('BLOCKED: current official index, downloaded wheel bytes or lock Torch hash mismatch.')
if system_site_packages_inventory(snapshot['systemSitePackagesRoots'])!=snapshot['systemSitePackagesFiles']: raise RuntimeError('BLOCKED: system site-packages changed before installation.')
venv_cfg=Path(venv_info['prefix'])/'pyvenv.cfg'; venv_cfg_sha=hashlib.sha256(venv_cfg.read_bytes()).hexdigest()
subprocess.run([VENV_PY,'-m','pip','install','--dry-run','--require-hashes','--report',str(plan),'-r',str(lock),'-c',str(REPO/'ai/constraints-colab.txt'),'--find-links',str(Path(wheel['wheelPath']).parent)],check=True,env=PIP_ENV)
if not validate_venv_install_plan(json.loads(plan.read_text()),torch_version='2.11.0+cu130',wheel_sha256=wheel['sha256'],verified_wheel_path=wheel['wheelPath'],python_executable=venv_info['pythonExecutable'],python_version=venv_info['python'],venv_prefix=venv_info['prefix'],base_prefix=venv_info['basePrefix'],torch_installed=venv_info['torch'] is not None,installed_packages=venv_info['installedPackages']): raise RuntimeError('BLOCKED: venv dry-run is malformed or replaces an already installed package.')
if verify_source_proof(source_proof_path,REPO)!=actual_commit or system_site_packages_inventory(snapshot['systemSitePackagesRoots'])!=snapshot['systemSitePackagesFiles']: raise RuntimeError('BLOCKED: source proof or system packages changed during dry-run.')
preinstall_gate={'verified':True,'sourceCommit':actual_commit,'sourceProofSha256':source_hash,'wheelSha256':wheel['sha256'],'lockSha256':hashlib.sha256(lock.read_bytes()).hexdigest(),'installPlanSha256':hashlib.sha256(plan.read_bytes()).hexdigest(),'systemSitePackagesSha256':hashlib.sha256(json.dumps(snapshot['systemSitePackagesFiles'],sort_keys=True).encode()).hexdigest(),'venvPythonExecutable':venv_info['pythonExecutable'],'venvPrefix':venv_info['prefix'],'venvBasePrefix':venv_info['basePrefix'],'venvCfgSha256':venv_cfg_sha,'venvInstalledPackages':venv_info['installedPackages']}
Path('/content/dacn-preinstall-gate.json').write_text(json.dumps(preinstall_gate,indent=2)+'\n')
print('PASS: clean source, official wheel SHA, full lock hash, isolated venv and Torch-only venv install plan. pip bootstrapped into isolated venv; locked application dependencies not installed.')


## Install into isolated venv after all pre-install gates pass

This cell changes only `/content/dacn-venv`. It exits before pip if source proof, wheel, lock, dry-run report or system package inventory has changed.

In [ ]:
from ai.colab.preflight import system_site_packages_inventory
from ai.colab.torch_provenance import verify_source_proof
preinstall_gate=json.loads(Path('/content/dacn-preinstall-gate.json').read_text())
lock=Path('/content/ai-requirements.lock'); plan=Path('/content/dacn-install-plan.json'); wheel=Path('/content/dacn-torch-wheel.json'); proof=Path('/content/dacn-source-proof.json')
wheel_info=json.loads(wheel.read_text()); proof_hash=hashlib.sha256(proof.read_bytes()).hexdigest(); actual_commit=verify_source_proof(proof,REPO)
if not preinstall_gate.get('verified') or actual_commit!=REVIEWED_COMMIT or preinstall_gate.get('sourceCommit')!=actual_commit or preinstall_gate.get('sourceProofSha256')!=proof_hash: raise RuntimeError('BLOCKED: source proof or pre-install gate changed.')
if hashlib.sha256(lock.read_bytes()).hexdigest()!=preinstall_gate.get('lockSha256') or hashlib.sha256(plan.read_bytes()).hexdigest()!=preinstall_gate.get('installPlanSha256'): raise RuntimeError('BLOCKED: lock or dry-run report changed after validation.')
if hashlib.sha256(Path(wheel_info['wheelPath']).read_bytes()).hexdigest()!=preinstall_gate.get('wheelSha256') or wheel_info.get('sha256')!=preinstall_gate.get('wheelSha256'): raise RuntimeError('BLOCKED: downloaded official Torch wheel changed.')
if system_site_packages_inventory(snapshot['systemSitePackagesRoots'])!=snapshot['systemSitePackagesFiles']: raise RuntimeError('BLOCKED: system site-packages changed before pip install.')
install_info=json.loads(subprocess.run([VENV_PY,'-B','-c',venv_probe],capture_output=True,text=True,check=True,env=PIP_ENV).stdout); install_cfg=Path(install_info['prefix'])/'pyvenv.cfg'
if VENV_PY!=preinstall_gate.get('venvPythonExecutable') or install_info['pythonExecutable']!=preinstall_gate.get('venvPythonExecutable') or install_info['prefix']!=preinstall_gate.get('venvPrefix') or install_info['basePrefix']!=preinstall_gate.get('venvBasePrefix') or install_info.get('torch') is not None or install_info['installedPackages']!=preinstall_gate.get('venvInstalledPackages') or 'include-system-site-packages = false' not in install_info['pyvenvCfg'].lower() or hashlib.sha256(install_cfg.read_bytes()).hexdigest()!=preinstall_gate.get('venvCfgSha256'): raise RuntimeError('BLOCKED: isolated venv identity/configuration changed after dry-run.')
PIP_ENV=isolated_venv_env()
SYSTEM_ENV=os.environ.copy()
for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE','VIRTUAL_ENV'): SYSTEM_ENV.pop(key,None)
SYSTEM_ENV['PYTHONDONTWRITEBYTECODE']='1'
subprocess.run([VENV_PY,'-m','pip','install','--require-hashes','-r',str(lock),'-c',str(REPO/'ai/constraints-colab.txt'),'--find-links',str(Path(wheel_info['wheelPath']).parent)],check=True,env=PIP_ENV)
if system_site_packages_inventory(snapshot['systemSitePackagesRoots'])!=snapshot['systemSitePackagesFiles']: raise RuntimeError('BLOCKED: system site-packages changed during venv installation.')
provenance_run=subprocess.run([sys.executable,'-B',str(REPO/'ai/colab/torch_provenance.py'),'--repo',str(REPO),'--snapshot','/content/dacn-colab-before.json','--source-proof','/content/dacn-source-proof.json','--output','/content/torch-wheel-provenance.json','--wheel',wheel_info['wheelPath'],'--venv-python',VENV_PY],capture_output=True,text=True,env=SYSTEM_ENV)
if provenance_run.stdout: print(provenance_run.stdout)
if provenance_run.returncode: raise RuntimeError(f'Torch venv provenance blocked (exit {provenance_run.returncode}):\n{provenance_run.stderr}')
provenance=json.loads(Path('/content/torch-wheel-provenance.json').read_text())
if provenance.get('sourceCommit')!=actual_commit or provenance.get('venvPythonExecutable')!=VENV_PY: raise RuntimeError('BLOCKED: venv provenance does not bind the reviewed source and target interpreter.')
print('PASS: dependencies installed in isolated venv; system inventory unchanged; official Torch RECORD matched.')

In [ ]:
# Partial preflight: modelLoad remains false until the venv runtime loads Qwen under the GPU lock.
from ai.colab.preflight import preflight_environment, system_site_packages_inventory
pip_check=subprocess.run([VENV_PY,'-m','pip','check'],capture_output=True,text=True,env=PIP_ENV)
if pip_check.returncode: raise RuntimeError('BLOCKED: venv pip check failed: '+pip_check.stdout+pip_check.stderr)
probe="import json,platform,subprocess,sys,torch,transformers,bitsandbytes; from pathlib import Path; cfg=(Path(sys.prefix)/'pyvenv.cfg').read_text().lower(); nf4=transformers.BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16); gpu=subprocess.run(['nvidia-smi','--query-gpu=name,uuid,memory.total','--format=csv,noheader'],capture_output=True,text=True,check=True).stdout.strip(); available=torch.cuda.is_available(); device=torch.cuda.get_device_name(0) if available else None; tensor=(torch.tensor([1.0,2.0],device='cuda')*2) if available else None; torch.cuda.synchronize() if available else None; cuda_test=bool(available and tensor.cpu().tolist()==[2.0,4.0]); print(json.dumps({'python':platform.python_version(),'pythonExecutable':sys.executable,'prefix':sys.prefix,'basePrefix':sys.base_prefix,'isolated':'include-system-site-packages = false' in cfg,'torch':torch.__version__,'torchRoot':str(Path(torch.__file__).parent.parent),'cuda':torch.version.cuda,'cudaAvailable':available,'cudaTest':cuda_test,'cudaDevice':device,'gpu':gpu,'nf4Import':isinstance(nf4,transformers.BitsAndBytesConfig)}))"
venv_probe=json.loads(subprocess.run([VENV_PY,'-B','-c',probe],capture_output=True,text=True,check=True,env=PIP_ENV).stdout)
if not venv_probe['cudaAvailable'] or not venv_probe['cudaTest'] or 'Tesla T4' not in str(venv_probe['cudaDevice']): raise RuntimeError('BLOCKED: venv PyTorch cannot execute a CUDA tensor operation on the Tesla T4.')
site_packages_after=system_site_packages_inventory(snapshot['systemSitePackagesRoots'])
checks={'python':venv_probe['python'],'torch':venv_probe['torch'],'cuda':venv_probe['cuda'],'gpu':venv_probe['gpu'].split(',')[0].strip(),
        'venvIsolated':venv_probe['isolated'] and venv_probe['prefix']!=venv_probe['basePrefix'],'systemSitePackagesUnchanged':site_packages_after==snapshot['systemSitePackagesFiles'],
        'systemSitePackagesBefore':snapshot['systemSitePackagesFiles'],'systemSitePackagesAfter':site_packages_after,
        'pipCheck':pip_check.returncode==0,'nf4Import':venv_probe['nf4Import'],'modelLoad':False}
venv_after={'environment':venv_probe,'pipCheck':pip_check.returncode==0,'pipFreeze':subprocess.run([VENV_PY,'-m','pip','freeze'],capture_output=True,text=True,check=True,env=PIP_ENV).stdout.splitlines(),'systemSitePackagesSha256':hashlib.sha256(json.dumps(site_packages_after,sort_keys=True,separators=(',',':')).encode()).hexdigest(),'modelLoad':False}
Path('/content/dacn-venv-after.json').write_text(json.dumps(venv_after,indent=2)+'\n')
partial=preflight_environment(checks); print(json.dumps(partial,indent=2))
if partial['reasons'] and any('modelLoad' not in reason for reason in partial['reasons']): raise RuntimeError('BLOCKED: isolated venv preflight failed.')

## Real Qwen model preflight and inference smoke

This cell loads the pinned model, verifies CUDA/NF4 and system-package preservation, then sends one synthetic `/v1/infer` smoke request. It does not run the 120-fixture benchmark.


In [ ]:
from getpass import getpass
import hashlib, time
from ai.colab.preflight import preflight_environment
os.environ['AI_RUNTIME_TOKEN']=getpass('Runtime bearer token (hidden): ')
os.environ['AI_RUNTIME_BENCHMARK_MODE']='0'
os.environ.pop('AI_RUNTIME_BENCHMARK_ALLOWLIST',None); os.environ.pop('AI_RUNTIME_OBSERVATION_DIR',None)
os.environ['AI_RUNTIME_MODEL_ID']='Qwen/Qwen3-4B-Instruct-2507'
os.environ['AI_RUNTIME_MODEL_REVISION']='cdbee75f17c01a7cc42f958dc650907174af0554'
os.environ['AI_RUNTIME_GPU_UUID']=venv_probe['gpu'].split(',')[1].strip()
PIP_ENV=isolated_venv_env()
server=subprocess.Popen([VENV_PY,'-m','uvicorn','ai.runtime.app:app','--host','127.0.0.1','--port','8000'],cwd=REPO,stdout=open('/content/dacn-runtime.log','w'),stderr=subprocess.STDOUT,env=PIP_ENV)
for _ in range(600):
    if server.poll() is not None: raise RuntimeError('Runtime/model startup failed; inspect local log.')
    log=Path('/content/dacn-runtime.log').read_text() if Path('/content/dacn-runtime.log').exists() else ''
    if 'Application startup complete' in log: break
    time.sleep(1)
else: server.terminate(); raise RuntimeError('Runtime/model startup timed out.')
site_packages_after=system_site_packages_inventory(snapshot['systemSitePackagesRoots'])
checks['systemSitePackagesAfter']=site_packages_after; checks['systemSitePackagesUnchanged']=site_packages_after==snapshot['systemSitePackagesFiles']; checks['modelLoad']=True
venv_after['systemSitePackagesUnchanged']=checks['systemSitePackagesUnchanged']; venv_after['modelLoad']=True; Path('/content/dacn-venv-after.json').write_text(json.dumps(venv_after,indent=2)+'\n')
result=preflight_environment(checks); print(json.dumps(result,indent=2))
if result['status']!='READY': server.terminate(); raise RuntimeError('BLOCKED: model/runtime preflight failed.')
Path('/content/dacn-model-preflight.json').write_text(json.dumps({'report':result,'checks':checks,'venv':venv_probe},indent=2)+'\n')
fixture_rows=[json.loads(line) for line in (REPO/'ai/benchmark/fixtures/fixtures.jsonl').read_text().splitlines()]
smoke_fixture=fixture_rows[0]; smoke_request={'contractVersion':'1.0','requestId':'phase1d3-real-inference-smoke','task':smoke_fixture['task'],'locale':'vi-VN','input':smoke_fixture['input']}
smoke_probe="import hashlib,httpx,json,os,sys; body=json.load(sys.stdin); response=httpx.post('http://127.0.0.1:8000/v1/infer',json=body,headers={'Authorization':'Bearer '+os.environ['AI_RUNTIME_TOKEN']},timeout=60); data=response.json(); valid=response.status_code==200 and data.get('requestId')==body['requestId'] and data.get('task')==body['task'] and data.get('runtime',{}).get('modelId')==os.environ['AI_RUNTIME_MODEL_ID'] and isinstance(data.get('output'),dict); print(json.dumps({'passed':valid,'statusCode':response.status_code,'requestId':data.get('requestId'),'task':data.get('task'),'modelId':data.get('runtime',{}).get('modelId'),'outputKeys':sorted(data['output']) if isinstance(data.get('output'),dict) else [],'responseSha256':hashlib.sha256(response.content).hexdigest()})); sys.exit(0 if valid else 1)"
smoke_run=subprocess.run([VENV_PY,'-B','-c',smoke_probe],input=json.dumps(smoke_request),capture_output=True,text=True,env=PIP_ENV)
if smoke_run.returncode: server.terminate(); raise RuntimeError('BLOCKED: real Qwen inference smoke failed: '+smoke_run.stderr)
smoke_evidence=json.loads(smoke_run.stdout); smoke_evidence.update({'sourceCommit':REVIEWED_COMMIT,'venvPythonExecutable':VENV_PY,'fixtureId':smoke_fixture['fixtureId']})
Path('/content/dacn-real-inference-smoke.json').write_text(json.dumps(smoke_evidence,indent=2)+'\n'); print(json.dumps(smoke_evidence,indent=2))
subprocess.run([VENV_PY,'ai/colab/measure_capacity.py'],cwd=REPO,check=True,env=PIP_ENV)
server.terminate(); server.wait(timeout=30)


## 120-fixture benchmark

Run only after the real inference smoke cell reports PASS. The benchmark restarts the runtime in observation mode so the smoke request is excluded from fixture measurements.


In [ ]:
if not result or result['status']!='READY': raise RuntimeError('Run the real-model preflight cell first and require READY.')
smoke_evidence=json.loads(Path('/content/dacn-real-inference-smoke.json').read_text())
if not smoke_evidence.get('passed') or smoke_evidence.get('sourceCommit')!=REVIEWED_COMMIT or smoke_evidence.get('venvPythonExecutable')!=VENV_PY: raise RuntimeError('BLOCKED: reviewed-source real inference smoke evidence is missing or mismatched.')
if verify_source_proof('/content/dacn-source-proof.json',REPO)!=REVIEWED_COMMIT or system_site_packages_inventory(snapshot['systemSitePackagesRoots'])!=snapshot['systemSitePackagesFiles']: raise RuntimeError('BLOCKED: source proof or system site-packages changed before benchmark.')
node_version=subprocess.run(['node','--version'],capture_output=True,text=True,check=True).stdout.strip().lstrip('v')
if not node_version.startswith('24.') or tuple(map(int,node_version.split('.')[:2]))<(24,18): raise RuntimeError('Node must be >=24.18.0 and <25.')
pnpm_version=subprocess.run(['pnpm','--version'],capture_output=True,text=True,check=True).stdout.strip()
if pnpm_version!='10.12.1': raise RuntimeError('pnpm must be 10.12.1.')
subprocess.run([VENV_PY,'ai/benchmark/validate_fixtures.py'],cwd=REPO,check=True,env=PIP_ENV)
fixture_rows=[json.loads(line) for line in (REPO/'ai/benchmark/fixtures/fixtures.jsonl').read_text().splitlines()]
from ai.benchmark.allowlist import fixture_request_hash
os.environ['AI_RUNTIME_BENCHMARK_MODE']='1'
os.environ['AI_RUNTIME_BENCHMARK_ALLOWLIST']=','.join(sorted({fixture_request_hash(row['task'],row['input']) for row in fixture_rows}))
os.environ['AI_RUNTIME_OBSERVATION_DIR']='/content/dacn-raw-observations'
os.environ['AI_RUNTIME_GPU_UUID']=venv_probe['gpu'].split(',')[1].strip()
os.environ['AI_BENCHMARK_GPU']=venv_probe['gpu'].split(',')[0].strip()
os.environ['AI_BENCHMARK_GPU_UUID']=venv_probe['gpu'].split(',')[1].strip()
os.environ['AI_BENCHMARK_TORCH']=venv_probe['torch']; os.environ['AI_BENCHMARK_CUDA']=venv_probe['cuda']
PIP_ENV=isolated_venv_env()
server=subprocess.Popen([VENV_PY,'-m','uvicorn','ai.runtime.app:app','--host','127.0.0.1','--port','8000'],cwd=REPO,stdout=open('/content/dacn-runtime.log','w'),stderr=subprocess.STDOUT,env=PIP_ENV)
for _ in range(600):
    if server.poll() is not None: raise RuntimeError('Benchmark runtime/model startup failed; inspect local log.')
    log=Path('/content/dacn-runtime.log').read_text() if Path('/content/dacn-runtime.log').exists() else ''
    if 'Application startup complete' in log: break
    time.sleep(1)
else: server.terminate(); raise RuntimeError('Benchmark runtime startup timed out.')
if system_site_packages_inventory(snapshot['systemSitePackagesRoots'])!=snapshot['systemSitePackagesFiles']: server.terminate(); raise RuntimeError('BLOCKED: system site-packages changed before benchmark dispatch.')
os.environ['AI_BENCHMARK_MODE']='1'
from ai.runtime.prompts import PROMPT_VERSIONS
os.environ['AI_BENCHMARK_CONFIG_HASH']=hashlib.sha256(json.dumps({'model':'Qwen/Qwen3-4B-Instruct-2507','revision':'cdbee75f17c01a7cc42f958dc650907174af0554','torch':venv_probe['torch'],'cuda':venv_probe['cuda'],'promptVersions':PROMPT_VERSIONS},sort_keys=True).encode()).hexdigest()
PIP_ENV=isolated_venv_env()
run_result=subprocess.run([VENV_PY,'ai/benchmark/run.py','--fixtures','ai/benchmark/fixtures/fixtures.jsonl','--endpoint','http://127.0.0.1:8000/v1/infer','--output-dir','/content/dacn-benchmark','--export-output','/content/dacn-benchmark/outputs'],cwd=REPO,capture_output=True,text=True,check=True,env=PIP_ENV)
run_dir=Path(json.loads(run_result.stdout)['runDir'])
print(f'Benchmark run directory: {run_dir}')


In [ ]:
# After manually adding scores.json, run this cell. It remains Pending until verified evidence.json is also present.
if 'run_dir' not in globals() or not (run_dir/'manifest.json').exists(): raise RuntimeError('Run the benchmark cell first; set run_dir to the intended resumed run when resuming.')
from ai.colab.preflight import system_site_packages_inventory
if system_site_packages_inventory(snapshot['systemSitePackagesRoots'])!=snapshot['systemSitePackagesFiles']: raise RuntimeError('BLOCKED: system site-packages changed during inference/benchmark.')
subprocess.run([VENV_PY,'ai/benchmark/report.py','--run-dir',str(run_dir)],cwd=REPO,check=True,env=PIP_ENV)
# Add one five-dimension score object per fixture to run_dir/scores.json before reporting final PASS eligibility.
import hashlib
artifact_roots=[Path('/content/dacn-benchmark'),Path('/content/dacn-real-inference-smoke.json'),Path('/content/dacn-model-preflight.json'),Path('/content/dacn-source-proof.json'),Path('/content/ai-requirements.lock'),Path('/content/dacn-torch-wheel.json'),Path('/content/dacn-preinstall-gate.json'),Path('/content/dacn-install-plan.json'),Path('/content/dacn-raw-observations'),Path('/content/dacn-capacity-measurements.json'),Path('/content/dacn-colab-before.json'),Path('/content/dacn-venv-after.json'),Path('/content/torch-wheel-provenance.json'),Path('/content/dacn-runtime.log')]
manifest={}
for item in artifact_roots:
    files=[item] if item.is_file() else ([f for f in item.rglob('*') if f.is_file()] if item.exists() else [])
    for file in files:
        if file.name!='SHA256SUMS.json': manifest[str(file)]=hashlib.sha256(file.read_bytes()).hexdigest()
Path('/content/dacn-phase1d3-SHA256SUMS.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('Evidence hash manifest: /content/dacn-phase1d3-SHA256SUMS.json')
import zipfile
archive='/content/dacn-phase1d3-evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
    for item in artifact_roots+[Path('/content/dacn-nestjs-transport-evidence.json')]:
        files=[item] if item.is_file() else ([f for f in item.rglob('*') if f.is_file()] if item.exists() else [])
        for file in files:
            if file.name!='SHA256SUMS.json': bundle.write(file,arcname=str(file).lstrip('/'))
    bundle.write('/content/dacn-phase1d3-SHA256SUMS.json',arcname='dacn-phase1d3-SHA256SUMS.json')
print('Return this archive:',archive)


## Return evidence

Review every output with ai/benchmark/rubric.md and save all five scores per attempted fixture. Return the full run directory, source hash, lock and wheel evidence, environment snapshot, fixture hash, attempts, evaluator results, scores, capacity measurements and NestJS transport evidence. Do not activate a profile or close OPEN-ARCH-003.